<a href="https://colab.research.google.com/github/StarxLoki/SFYouTubeCode/blob/main/Copy_of_Anime_Recommdation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
organizations_cooperunion_anime_recommendations_database_path = kagglehub.dataset_download('organizations/CooperUnion/anime-recommendations-database')
qwen_lm_qwen_3_transformers_0_6b_base_1_path = kagglehub.model_download('qwen-lm/qwen-3/Transformers/0.6b-base/1')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **📖 Anime Adventure Blog: How We Built a Smart Recommendation Bot**


## **🧠 Model - Qwen3**
Qwen3 is a large language model (LLM) developed by Alibaba Cloud, part of the Qwen series — one of the most powerful open-source LLMs available today. It's trained on massive amounts of text data from the web, books, code, and more.

## 🔍 Key Features of Qwen3:
| Feature                     | Description                                                                     |
|-----------------------------|---------------------------------------------------------------------------------|
| Multilingual Support        | Understands and generates text in multiple languages including English, Chinese, Japanese, Korean, etc. |
| Code Writing & Explanation  | Write scripts, debug, or explain programming logic.                             |
| Dialogue Understanding      | Built to understand and generate conversationally natural responses.            |
| Creative Writing            | Generate stories, poems, screenplays, and fictional worlds.                     |
| Reasoning & Logic           | Solve puzzles, math problems, and logical questions.                             |
| Fine-tuned for Tasks        | Some versions are fine-tuned for specific domains like storytelling, coding, and chatbots. |


## **📌 Task Outline: What This Notebook Does**

The notebook you're building leverages Qwen3’s capabilities in combination with a real anime recommendation dataset, allowing users to:

- Understand Anime Trends
- Generate Personalized Anime Recommendations
- Combine Data Filtering with AI Creativity
- Anime Recommendation depends on out mood using Qwen Thinking mode.

## 🔧 Setup & Dependencies

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


## 🔑 Load Qwen3 Model and Tokenizer


In [ ]:
# Load Qwen 0.6B model
model_path = "/kaggle/input/qwen-3/transformers/0.6b-base/1"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).to("cuda" if torch.cuda.is_available() else "cpu")

# Load dataset
df = pd.read_csv("/kaggle/input/anime-recommendations-database/anime.csv")
print(df.head(5))

## 📊 Explore the Anime Dataset


In [ ]:
#Filter and visualize the dataset
print("Top 5 Highest-Rated Animes:")
display(df.sort_values("rating", ascending=False).head()[["name", "genre", "type", "rating"]])

plt.figure(figsize=(10, 6))
sns.countplot(y="type", data=df, order=df["type"].value_counts().index[:5])
plt.title("Top 5 Anime Types (Movie/Series/etc.)")
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(df["rating"], bins=20, kde=True)
plt.title("Distribution of Anime Ratings")
plt.xlabel("Rating (Out of 10)")
plt.ylabel("Count")
plt.show()

## 🤖 Anime Recommendation Bot
### Helper Function for Model Inference
Using Qwen 0.6B to generate personalized anime suggestions


In [ ]:
def qwen_generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id ,
        temperature=0.7,
        do_sample=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## Use Qwen 0.6B to generate personalized anime suggestions

## 1: Genre-Based Recommendations



In [ ]:
prompt = """Suggest 3 action anime for someone who loves complex villains.
Explain why each matches the request. Use emojis."""
response = qwen_generate(prompt)
print(response)

## 2. Underrated Gems


In [ ]:
prompt = "List 3 underrated anime with unique art styles. Include genres and summaries."
print(qwen_generate(prompt))

## 3. Analyze Trends with Qwen


In [ ]:
prompt = """The dataset shows 40% of top-rated animes are action-themed.
Explain why action anime dominates global popularity. List 3 cultural/historical reasons."""
analysis = qwen_generate(prompt)
print(analysis)

#### *From initial Qwen3 interactions based on general knowledge, now we will using the dataset with qwen3 mode. This shift demonstrably enhanced the LLM's ability to provide targeted and insightful anime recommendations, highlighting the strength of data-augmented LLM applications.*
## **🤖 LLM + Dataset Interaction**
### 1. Filter Recommendations Using Data - Combine dataset filters with AI creativity!



In [ ]:
# **Convert the 'episodes' column to numeric if it's not already**
if df['episodes'].dtype == 'object':
    df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
    # Remove rows where conversion failed (NaN values in 'episodes')
    df.dropna(subset=['episodes'], inplace=True)

# Filter dataset for short series (<=12 episodes)
short_series = df[(df["episodes"] <= 12) & (df["episodes"] > 1)].sort_values("rating", ascending=False)

#Prompting with dataset results
prompt = f"""Based on this list of short anime:\n{short_series.head(5)[['name', 'genre', 'rating']]}\n\n
Suggest 2 binge-worthy short series and explain why they're worth watching."""
print(qwen_generate(prompt))

## 2. Recommend Based on Mood + Dataset
#### Here, we leverage Qwen3's reasoning capabilities, evident in its self-generated "thinking" process, to understand user-stated moods. By grounding this inherent understanding with a curated list of anime titles, genres, and ratings from our dataset, we move beyond generic suggestions. This approach empowers Qwen3 to make more contextually relevant and data-backed recommendations, demonstrating the synergy between its intrinsic knowledge processing and the power of external information.


In [ ]:
def recommend_anime_with_context_direct(mood, top_k=5):
    # Build prompt context from dataset
    context = "\n".join([
        f"{row['name']} ({row['genre']}): Rating {row['rating']}"
        for _, row in df.sort_values('rating', ascending=False).head(20).iterrows()
    ])

    prompt = f"""
<thinking>
Okay, the user is feeling "{mood}". I need to recommend some anime that match this emotional state.
First, I'll look at the genres typically associated with this mood — for example, melancholic moods might align with Drama or Tragedy.
Next, I'll scan the provided list of anime and check their genres and description.
I should prioritize ones with high scores and positive community feedback.
Finally, I'll summarize each recommendation clearly, explaining why it fits the mood.
</thinking>

Based on the following anime list, recommend up to {top_k} titles that best fit someone feeling "{mood}".

Anime List:
{context}

Recommendations:
"""
    result = qwen_generate(prompt)
    print("Recommendations:")
    print(result.replace('\n', '\n  ')) # Adding indentation for better readability
    return result

# Call func
#recommend_anime_with_context_direct(input("Please Enter your mood, I will recommend top anime for you"))
recommend_anime_with_context_direct('Action')


### From Anime Chat to Data-Informed Insights
Witness how structured anime data transforms Qwen3's understanding, moving beyond simple interaction to insightful analysis and targeted recommendations.

| **🎯 What You’re Missing If You Skip the Dataset?** | **With Dataset** | **Without Dataset** |
|-----------------------------------------------|--------------|-----------------|
| Filter by genre/rating/episode count          | ✅ Yes       | ❌ No           |
| Show actual stats (e.g., "Top 5 rated")       | ✅ Yes       | ❌ Hardcoded or generic |
| Build personalized filters                     | ✅ Yes       | ❌ Very limited   |
| Add interactivity (e.g., UI)                 | ✅ Yes       | ❌ Harder to justify results |


### What actually done in this notebook
- Qwen is the Storyteller & Recommender – creative, expressive, knows a lot.
- Dataset is the Library – full of facts, lists, numbers, and structure.

You wouldn’t ask a storyteller to guess what books are in a library — instead, show them the shelves and say:
"From these books, which would you recommend to a fantasy fan?"
That’s exactly what our notebook does!
